# 01_1 · Ampliación dirigida de categorías de daño

Este cuaderno reproduce la ampliación iniciada desde el flujo de los cuadernos 01–04 sin modificar el corpus canónico mientras sigue abierta la adjudicación humana original. El lote queda aislado en `datos/ampliacion/ampliacion_dano_20260726`.

El objetivo de muestreo es enriquecer las cinco **categorías gruesas** y reducir su desbalance. Las etiquetas finas se usan para mantener la coherencia del etiquetado, y los flags transversales para trazabilidad/enrutamiento; ninguno se convierte en objetivo de entrenamiento. El muestreo dirigido no permite estimar prevalencias poblacionales.

## Flujo y salvaguardas

1. Descubrir videos de fuentes históricamente productivas y mediante consultas orientadas a categorías minoritarias.
2. Descargar únicamente subtítulos públicos VTT; no se descarga audio ni video.
3. Segmentar y deduplicar con reglas compatibles con el cuaderno 02.
4. Etiquetar todos los chunks con Flash.
5. Enviar a Pro todo daño Flash, toda duda, confianza menor que 0.90 y un control determinista del 10% de seguros confiables.
6. Incorporar solo decisiones Pro resueltas; separar `needs_review=True` de Pro en una cola humana.
7. Añadir los casos nuevos únicamente a train/validation por grupos de `video_id`; mantener congelado el test histórico.
8. Reentrenar solo cuando las 139 adjudicaciones humanas originales estén completas.

> Las celdas que consumen red o API están desactivadas de forma predeterminada. Los scripts son reanudables, pero volver a ejecutarlos puede descubrir candidatos distintos o generar costo.

In [ ]:
from pathlib import Path
import json
import subprocess
import sys

ROOT = Path.cwd().resolve()
if ROOT.name.lower() == 'cuadernos':
    ROOT = ROOT.parent
BATCH = ROOT / 'datos' / 'ampliacion' / 'ampliacion_dano_20260726'
REPORT = ROOT / 'resultados' / 'INFORME_AMPLIACION_DIRIGIDA_DANO.md'
print('Raíz:', ROOT)
print('Lote:', BATCH)

## 1. Adquisición y segmentación (cuadernos 01 y 02)

Cada fase escribe artefactos y manifiestos en el lote aislado. Cambie a `True` solo la fase que desee repetir.

In [ ]:
RUN_DISCOVERY = False
RUN_TRANSCRIPTS = False
RUN_CHUNKS = False

for enabled, stage in [(RUN_DISCOVERY, 'discover'), (RUN_TRANSCRIPTS, 'transcribe'), (RUN_CHUNKS, 'chunk')]:
    if enabled:
        subprocess.run(
            [sys.executable, '-m', 'scripts_auxiliares.ampliacion_dirigida_dano', '--stage', stage],
            cwd=ROOT, check=True,
        )
    else:
        print(f'{stage}: omitido (artefactos existentes preservados)')

## 2. Etiquetado Flash → Pro (cuaderno 03_2)

Flash procesa todo el lote. Pro revisa los ejemplos seleccionados por la regla de duda/daño/control. La validación estricta del esquema impide escribir respuestas incompletas. Las llamadas son reanudables por `chunk_id`.

In [ ]:
RUN_FLASH = False  # consume API; ya completado para 21.991 chunks
RUN_PRO = False    # consume API; ya completado para 5.183 chunks

for enabled, stage in [(RUN_FLASH, 'flash'), (RUN_PRO, 'pro')]:
    if enabled:
        subprocess.run(
            [sys.executable, '-m', 'scripts_auxiliares.etiquetar_ampliacion_dano', '--stage', stage],
            cwd=ROOT, check=True,
        )
    else:
        print(f'{stage}: omitido (no se realizarán llamadas)')

## 3. Dataset utilizable, balance y estimación

Esta fase es local, no consume API y puede repetirse. Comprueba cardinalidades e identificadores, excluye dudas Pro y genera la partición nueva por video, el gráfico y el informe Markdown.

In [ ]:
subprocess.run(
    [sys.executable, '-m', 'scripts_auxiliares.preparar_entrenamiento_ampliado', '--stage', 'prepare'],
    cwd=ROOT, check=True,
)

In [ ]:
manifest_path = BATCH / 'processed' / 'dataset_etiquetado_utilizable.manifest.json'
manifest = json.loads(manifest_path.read_text(encoding='utf-8'))
print(json.dumps({
    'usable': manifest['usable_rows'],
    'pending_human': manifest['pending_human_rows'],
    'split': manifest['split'],
    'balance': manifest['balance'],
    'threat_estimate': manifest['threat_video_estimate'],
}, ensure_ascii=False, indent=2))

## 4. Reentrenamiento del moderador (cuaderno 04)

La siguiente celda está intencionalmente desactivada. El script verifica que las 139 decisiones humanas originales estén completas; si no lo están, aborta antes de entrenar. Compara el baseline reproducido, el conjunto ampliado sin AEDA y el ampliado con AEDA. La selección usa la validación histórica y el test histórico permanece congelado.

In [ ]:
RUN_TRAINING = False
if RUN_TRAINING:
    subprocess.run(
        [sys.executable, '-m', 'scripts_auxiliares.preparar_entrenamiento_ampliado', '--stage', 'train'],
        cwd=ROOT, check=True,
    )
else:
    print('Entrenamiento omitido. Actívelo después de completar 139/139 casos humanos.')

## Resultado actual

El informe reproducible, sus cálculos, limitaciones y referencias APA 7 están en `resultados/INFORME_AMPLIACION_DIRIGIDA_DANO.md`. Al finalizar el entrenamiento, el mismo archivo se actualiza con la comparación de modelos y el modelo exportado.